# Geopolitical Exposure Proof of Concept

## 00 — Research design and pre-outcome protocol

This notebook defines a transcript-ready experiment connecting point-in-time corporate text, holdings-based asset embeddings, investor heterogeneity, and subsequent geopolitical event reactions. It creates design manifests only; it does not inspect event returns, select seed firms, or call an LLM.


In [1]:
from pathlib import Path
import os,sys,json,hashlib,platform
import pandas as pd,numpy as np
from IPython.display import display
HERE=Path.cwd().resolve()
if HERE.name=='geopolitical_exposure': ROOT=HERE.parent
elif HERE.name=='Asset_Embeddings': ROOT=HERE
else: ROOT=(HERE/'Projects'/'Asset_Embeddings').resolve()
EXT=ROOT/'geopolitical_exposure'; CONFIG=EXT/'config'; SCHEMAS=EXT/'schemas'; MANIFESTS=EXT/'manifests'
for path in [CONFIG,SCHEMAS,MANIFESTS]: path.mkdir(parents=True,exist_ok=True)
def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as handle:
  for block in iter(lambda:handle.read(1024*1024),b''): h.update(block)
 return h.hexdigest()
print('Project root:',ROOT)


Project root: /Users/harmandeep/Desktop/Projects/Asset_Embeddings


## 1. Verify the July 2026 paper

The attached paper is treated as research input, not as instructions. We verify its identity and record the pages motivating this extension. The July 2026 paper uses earnings calls to interpret asset neighborhoods, investor strategy descriptions to interpret investor neighborhoods, and externally chosen exemplar firms for generative portfolios. Our extension focuses on the unresolved exemplar-selection and point-in-time validation problem.


In [2]:
package_root=ROOT/'.packages'
if package_root.exists() and str(package_root) not in sys.path: sys.path.insert(0,str(package_root))
from pypdf import PdfReader
paper_path=ROOT/'ssrn-5212742.pdf'; assert paper_path.exists()
reader=PdfReader(paper_path); metadata=reader.metadata
paper_audit=pd.DataFrame([{'file':paper_path.name,'title':metadata.get('/Title'),'authors':metadata.get('/Author'),'creation_date':metadata.get('/CreationDate'),'pages':len(reader.pages),'sha256':sha256_file(paper_path)}])
assert paper_audit.title.iloc[0]=='Asset and Investor Embeddings' and paper_audit.pages.iloc[0]==64
paper_routes=pd.DataFrame([
 {'page':5,'topic':'Earnings-call and investor-text interpretation','role_in_extension':'Motivates a modular text interpretation layer'},
 {'page':10,'topic':'Investor heterogeneity and disagreement','role_in_extension':'Motivates comparison across investor groups'},
 {'page':42,'topic':'LLM summaries of earnings calls','role_in_extension':'Motivates evidence-grounded structured extraction'},
 {'page':46,'topic':'Generative portfolios and COVID exemplars','role_in_extension':'Motivates scenario-to-seed-to-neighbor construction'},
 {'page':47,'topic':'Investor characteristics including Active Share and turnover','role_in_extension':'Clarifies distinction from the investor-identification experiment'}])
display(paper_audit); display(paper_routes)


,file,title,authors,creation_date,pages,sha256
0,ssrn-5212742.pdf,Asset and Investor Embeddings,Xavier Gabaix Ralph S.J. Koijen Robert J. Rich...,D:20260716132917-04'00',64,201ccc3c5ce18bc91535068b241ea27032320ad3de8feb...


,page,topic,role_in_extension
0,5,Earnings-call and investor-text interpretation,Motivates a modular text interpretation layer
1,10,Investor heterogeneity and disagreement,Motivates comparison across investor groups
2,42,LLM summaries of earnings calls,Motivates evidence-grounded structured extraction
3,46,Generative portfolios and COVID exemplars,Motivates scenario-to-seed-to-neighbor constru...
4,47,Investor characteristics including Active Shar...,Clarifies distinction from the investor-identi...


## 2. Primary research question and hypotheses

> Can pre-event corporate text identify direct geopolitical exposures, and can holdings-based embeddings propagate those signals to non-seed firms whose exposure is not obvious in their own disclosures?

The primary analysis excludes all direct seed firms. This prevents the embedding model from receiving credit for firms already identified by text.


In [3]:
hypotheses=pd.DataFrame([
 ('H1','Direct text exposure','Pre-event text scores covary with post-event abnormal returns in the predicted direction.'),
 ('H2','Incremental embedding information','Embedding exposure adds explanatory power beyond text, industry, and frozen firm controls among non-seed firms.'),
 ('H3','Latent-exposure quadrant','Low-text/high-embedding non-seed firms react more strongly than low-text/low-embedding controls.'),
 ('H4','Investor heterogeneity','High-Active-Share embeddings propagate exposure more effectively than passive-like embeddings from identical seeds.'),
 ('H5','Combined model','A frozen text-plus-embedding score outperforms text alone without outcome-tuned weights.')],columns=['hypothesis','test','prediction'])
display(hypotheses)


,hypothesis,test,prediction
0,H1,Direct text exposure,Pre-event text scores covary with post-event a...
1,H2,Incremental embedding information,Embedding exposure adds explanatory power beyo...
2,H3,Latent-exposure quadrant,Low-text/high-embedding non-seed firms react m...
3,H4,Investor heterogeneity,High-Active-Share embeddings propagate exposur...
4,H5,Combined model,A frozen text-plus-embedding score outperforms...


## 3. Candidate pilot and channel decomposition

The Russian invasion of Ukraine on February 24, 2022 is a candidate pilot, not yet an accepted event. It is accepted only if the data audit passes using coverage and timing criteria that do not inspect event returns. All exposure channels must be frozen before text extraction.


In [4]:
with open(CONFIG/'russia_ukraine_pilot.json') as handle: event_config=json.load(handle)
channels=pd.DataFrame(event_config['channels']); display(pd.DataFrame([{k:v for k,v in event_config.items() if k!='channels'}])); display(channels)


,status,event_id,event_name,event_date,information_cutoff,preferred_holdings_snapshot,primary_outcome_windows,estimation_window,selection_rule
0,candidate_pending_data_audit,russia_ukraine_invasion_2022,Russian invasion of Ukraine,2022-02-24,2022-02-23,2021Q4,"[[0, 1], [0, 5], [1, 20]]","[-250, -30]","Finalize only if pre-event text, 2021Q4 embedd..."


,channel,expected_direction
0,russia_ukraine_operations_revenue,negative
1,european_energy_dependence,negative
2,oil_gas_production,positive
3,defense_demand,positive
4,agriculture_fertilizer,ambiguous
5,european_banking_sanctions,negative
6,shipping_logistics,negative
7,sanctions_sensitive_supply_chain,negative


## 4. Frozen comparison models

Every embedding model receives exactly the same positive and negative text-derived seeds. The primary propagation test uses 10-dimensional RS-Binary embeddings because that representation already exists across all investor groups. Four-dimensional projections are a preregistered low-dimensional robustness test. Exact portfolio weights are secondary.


In [5]:
models=pd.DataFrame([
 ('text_only','Direct structured text exposure; no embedding expansion','benchmark'),
 ('all_s12_binary','2021Q4 all-S12 RS-Binary embedding expansion','primary embedding'),
 ('stable_stock_picker_binary','High Active Share / low turnover expansion','investor mechanism'),
 ('tactical_active_binary','High Active Share / high turnover expansion','investor mechanism'),
 ('benchmark_rebalancer_binary','Low Active Share / high turnover expansion','investor mechanism'),
 ('passive_like_binary','Low Active Share / low turnover expansion','investor mechanism'),
 ('combined_text_embedding','Frozen text and embedding scores','combined model'),
 ('industry_characteristics','Industry plus pre-event firm controls','conventional benchmark')],columns=['model','definition','role'])
display(models)


,model,definition,role
0,text_only,Direct structured text exposure; no embedding ...,benchmark
1,all_s12_binary,2021Q4 all-S12 RS-Binary embedding expansion,primary embedding
2,stable_stock_picker_binary,High Active Share / low turnover expansion,investor mechanism
3,tactical_active_binary,High Active Share / high turnover expansion,investor mechanism
4,benchmark_rebalancer_binary,Low Active Share / high turnover expansion,investor mechanism
5,passive_like_binary,Low Active Share / low turnover expansion,investor mechanism
6,combined_text_embedding,Frozen text and embedding scores,combined model
7,industry_characteristics,Industry plus pre-event firm controls,conventional benchmark


## 5. Temporal and leakage rules

The event outcome is quarantined until the document corpus, prompts, channels, direct seeds, thresholds, embedding dimensions, scoring rules, and event windows are frozen. A filing's `available_date`, not fiscal-period end, determines eligibility. Holdings are assigned using their public availability date and reporting lag. No transformation fitted on post-event quarters may enter the pilot.


In [6]:
rules=pd.DataFrame([
 ('text_cutoff','Only documents with available_date <= 2022-02-23.'),('holdings_cutoff','Use only holdings publicly available before the event; audit 2021Q4 reporting availability.'),('embedding_fit','Use event-specific pre-event embeddings; no future-quarter fitting or normalization.'),('seed_freeze','Freeze evidence-backed seeds before loading event outcomes.'),('seed_exclusion','Exclude all direct seed firms from the primary propagation evaluation.'),('model_freeze','Freeze prompt, model ID, schema, retrieval rules, dimensions, and thresholds.'),('outcome_quarantine','Do not inspect event returns during event/channel/text/model selection.'),('identifier_history','Use point-in-time company, PERMNO, and PERMCO links.'),('negative_evidence','Retain no-exposure findings, refusals, failures, and ambiguous directions.'),('placebos','Use placebo dates, permuted seeds, pre-trends, and alternative event windows.')],columns=['rule','requirement'])
display(rules)


,rule,requirement
0,text_cutoff,Only documents with available_date <= 2022-02-23.
1,holdings_cutoff,Use only holdings publicly available before th...
2,embedding_fit,Use event-specific pre-event embeddings; no fu...
3,seed_freeze,Freeze evidence-backed seeds before loading ev...
4,seed_exclusion,Exclude all direct seed firms from the primary...
5,model_freeze,"Freeze prompt, model ID, schema, retrieval rul..."
6,outcome_quarantine,Do not inspect event returns during event/chan...
7,identifier_history,"Use point-in-time company, PERMNO, and PERMCO ..."
8,negative_evidence,"Retain no-exposure findings, refusals, failure..."
9,placebos,"Use placebo dates, permuted seeds, pre-trends,..."


## 6. Transcript-ready document and extraction contracts

SEC filings and earnings-call transcripts normalize to the same document schema. The extraction schema requires bounded scores, direction, confidence, and verbatim evidence offsets. This makes the eventual transcript substitution a data-adapter change rather than a redesign.


In [7]:
with open(SCHEMAS/'document_schema.json') as handle: document_schema=json.load(handle)
with open(SCHEMAS/'exposure_output_schema.json') as handle: exposure_schema=json.load(handle)
schema_audit=pd.DataFrame([{'schema':'document','required_fields':len(document_schema['required']),'sha256':sha256_file(SCHEMAS/'document_schema.json')},{'schema':'exposure_output','required_fields':len(exposure_schema['required']),'sha256':sha256_file(SCHEMAS/'exposure_output_schema.json')}])
display(schema_audit)


,schema,required_fields,sha256
0,document,7,f3e66768bfeecd6351fe3b0baea4e22a809443df27db5c...
1,exposure_output,5,25f76e989422ab67fa1a7d7134b84eb20d74bada127614...


## 7. Freeze the pre-audit design

This hash freezes the design inputs before the event/data audit. It does not freeze direct seeds or accept the candidate event. Those require separate manifests after their respective gates pass.


In [8]:
design_payload={'stage':'pre_event_data_audit','outcomes_inspected':False,'event_status':event_config['status'],'paper_sha256':paper_audit.sha256.iloc[0],'event_config_sha256':sha256_file(CONFIG/'russia_ukraine_pilot.json'),'document_schema_sha256':schema_audit.loc[schema_audit.schema.eq('document'),'sha256'].iloc[0],'exposure_schema_sha256':schema_audit.loc[schema_audit.schema.eq('exposure_output'),'sha256'].iloc[0],'hypotheses':hypotheses.to_dict(orient='records'),'models':models.to_dict(orient='records'),'temporal_rules':rules.to_dict(orient='records')}
payload_text=json.dumps(design_payload,sort_keys=True); design_payload['sha256']=hashlib.sha256(payload_text.encode()).hexdigest()
with open(MANIFESTS/'research_design_pre_audit.json','w') as handle: json.dump(design_payload,handle,indent=2)
print('Pre-audit research design SHA-256:',design_payload['sha256'])
print('Next: run 01_Event_and_Data_Audit.ipynb. Do not load event outcomes yet.')


Pre-audit research design SHA-256: 9016570954552ba07eb199ce8cab392536e6dd52e515f26b9184b3f0ed284703
Next: run 01_Event_and_Data_Audit.ipynb. Do not load event outcomes yet.
